# Feature Builder

This notebook is strictly for feature creation and feature plots. It does not build targets, correlations, IC tables, or distribution diagnostics.

Default bucket size is `BAR_MINUTES = 5`. Override with `MODL_BAR_MINUTES=1` or `MODL_BAR_MINUTES=15` before launching Jupyter.

In [ ]:
from __future__ import annotations

import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns

if "notebooks" not in sys.path:
    sys.path.append("notebooks")

from advanced_features import (
    build_feature_set,
    discover_datasets,
    latest_feature_date,
    plot_realized_vol,
    plot_term_structure,
)

ROOT = Path(os.environ.get("MODL_WS_NORMALIZED_DIR", "/mnt/burner-archive/ws_normalized")).expanduser()
DATE = os.environ.get("MODL_VIEW_DATE") or latest_feature_date(ROOT)
DATE_TAG = datetime.strptime(DATE, "%Y-%m-%d").strftime("%y-%m-%d")
FEATURE_ROOT = Path(os.environ.get("MODL_WS_FEATURE_DIR", "/mnt/burner-archive/ws_features")).expanduser()
BAR_MINUTES = int(os.environ.get("MODL_BAR_MINUTES", "5"))
HORIZONS = (5, 15, 30)
SAVE_OUTPUTS = False

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 240)
pd.set_option("display.max_colwidth", 180)
pl.Config.set_tbl_cols(240)
pl.Config.set_tbl_rows(24)

DATASETS = discover_datasets(ROOT, DATE_TAG)
if not DATASETS:
    raise FileNotFoundError(f"No normalized Parquet files found under {ROOT} for {DATE}")

ROOT, DATE, DATE_TAG, BAR_MINUTES, len(DATASETS)


## Build 5-Minute Feature Set

In [ ]:
feature_set = build_feature_set(DATASETS, horizons=HORIZONS, bar_minutes=BAR_MINUTES)
feature_matrix = feature_set.feature_matrix
base_feature_matrix = feature_set.base_feature_matrix
trade_features = feature_set.trade_features
book_features = feature_set.book_features
deribit_option_features = feature_set.deribit_option_features
term_structure_features = feature_set.term_structure
option_smile_features = feature_set.option_smile
futures_basis_features = feature_set.futures_basis
funding_features = feature_set.funding_features
rv_features = feature_set.rv_features
hawkes_bsi_features = feature_set.hawkes_bsi_features
reference_price = feature_set.reference_price

df = feature_matrix
df_stats = df.describe().T

component_shapes = pd.DataFrame(
    [
        ("feature_matrix", *feature_matrix.shape),
        ("base_feature_matrix", *base_feature_matrix.shape),
        ("trade_features", trade_features.height, trade_features.width),
        ("book_features", book_features.height, book_features.width),
        ("deribit_option_features", *deribit_option_features.shape),
        ("term_structure_features", *term_structure_features.shape),
        ("option_smile_features", *option_smile_features.shape),
        ("futures_basis_features", *futures_basis_features.shape),
        ("funding_features", *funding_features.shape),
        ("rv_features", *rv_features.shape),
        ("hawkes_bsi_features", *hawkes_bsi_features.shape),
    ],
    columns=["component", "rows", "columns"],
)
component_shapes

In [ ]:
df_stats.head(80)

## Trade Flow Features

In [ ]:
trade_features.head(30)

In [ ]:
trade_pdf = trade_features.to_pandas()
fig, axes = plt.subplots(4, 1, figsize=(15, 14), sharex=True)
sns.lineplot(data=trade_pdf, x="minute", y="vwap", hue="venue", marker="o", ax=axes[0])
axes[0].set_title(f"{BAR_MINUTES}-Minute VWAP")
sns.lineplot(data=trade_pdf, x="minute", y="trade_count", hue="venue", marker="o", ax=axes[1], legend=False)
axes[1].set_title("Trade Count")
sns.lineplot(data=trade_pdf, x="minute", y="volume", hue="venue", marker="o", ax=axes[2], legend=False)
axes[2].set_title("Trade Volume")
sns.lineplot(data=trade_pdf, x="minute", y="flow_imbalance", hue="venue", marker="o", ax=axes[3], legend=False)
axes[3].axhline(0, color="black", linewidth=1)
axes[3].set_title("Flow Imbalance")
fig.autofmt_xdate(); plt.tight_layout()

## Book And Quote Features

In [ ]:
book_features.head(30)

In [ ]:
book_pdf = book_features.to_pandas()
fig, axes = plt.subplots(3, 1, figsize=(15, 12), sharex=True)
sns.lineplot(data=book_pdf, x="minute", y="mid", hue="venue", marker="o", ax=axes[0])
axes[0].set_title("Mid Price")
sns.lineplot(data=book_pdf, x="minute", y="spread_bps", hue="venue", marker="o", ax=axes[1], legend=False)
axes[1].set_title("Spread Bps")
imbalance_pdf = book_pdf.melt(
    id_vars=["minute", "venue"],
    value_vars=[column for column in ["top_imbalance", "depth_imbalance"] if column in book_pdf.columns],
    var_name="metric",
    value_name="imbalance",
).dropna()
sns.lineplot(data=imbalance_pdf, x="minute", y="imbalance", hue="venue", style="metric", marker="o", ax=axes[2])
axes[2].axhline(0, color="black", linewidth=1)
axes[2].set_title("Book Imbalance")
fig.autofmt_xdate(); plt.tight_layout()

## Realized Volatility And Bipower Variation Features

In [ ]:
rv_features.tail(30)

In [ ]:
plot_realized_vol(rv_features)

## Deribit IV Term Structure Features

In [ ]:
term_structure_features.tail(30)

In [ ]:
plot_term_structure(term_structure_features)

## Deribit Option Smile Features

In [ ]:
option_smile_features.tail(30)

In [ ]:
smile_cols = [column for column in ["smile_otm_put_iv", "smile_atm_put_iv", "smile_atm_call_iv", "smile_otm_call_iv"] if column in option_smile_features]
fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
option_smile_features[smile_cols].plot(ax=axes[0], marker="o")
axes[0].set_title("Option Smile IV Buckets")
smile_spread_cols = [column for column in option_smile_features.columns if column.endswith("proxy") or column.endswith("minus_atm")]
option_smile_features[smile_spread_cols].plot(ax=axes[1], marker="o")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Smile Skew / Butterfly Proxies")
fig.autofmt_xdate(); plt.tight_layout()

## Deribit Futures Basis Features

In [ ]:
futures_basis_features.tail(30)

In [ ]:
basis_cols = [column for column in futures_basis_features.columns if column.startswith("basis_fut_") and not column.endswith("count") and not column.endswith("annualized")]
ann_cols = [column for column in futures_basis_features.columns if column.endswith("annualized")]
fig, axes = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
futures_basis_features[basis_cols].plot(ax=axes[0], marker="o")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Deribit Futures Basis Bps")
futures_basis_features[ann_cols].plot(ax=axes[1], marker="o")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Annualized Futures Basis Bps")
fig.autofmt_xdate(); plt.tight_layout()

## Funding Features

In [ ]:
funding_features.tail(30)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
funding_features["estimated_funding_rate"].plot(ax=ax, marker="o")
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Hibachi Estimated Funding Rate")
plt.tight_layout()

## Hawkes Buy/Sell Imbalance Features

These features treat signed trade volume as a self-exciting order-flow process. For each venue, `signed_volume = buy_volume - sell_volume` is passed through an exponential Hawkes-style memory filter: `state[t] = exp(-kappa) * state[t-1] + signed_volume[t]`.

`hawkes_bsi_k010_*`, `hawkes_bsi_k030_*`, and `hawkes_bsi_k050_*` are raw decayed signed-volume pressure at slow, medium, and fast decay speeds. `hawkes_bsi_norm_*` divides that pressure by decayed total volume so venues are comparable. Cross-venue mean, standard deviation, range, and positive-share columns summarize whether buying or selling pressure is broad or venue-specific.

In [ ]:
hawkes_bsi_features.tail(30)

In [ ]:
hawkes_plot_cols = [
    column for column in hawkes_bsi_features.columns
    if column.startswith("hawkes_bsi_norm_k030_")
    and not column.startswith("hawkes_bsi_norm_diff_")
    and column not in {
        "hawkes_bsi_norm_mean_k030",
        "hawkes_bsi_norm_std_k030",
        "hawkes_bsi_norm_range_k030",
        "hawkes_bsi_norm_positive_share_k030",
    }
]
aggregate_cols = [
    column for column in [
        "hawkes_bsi_norm_mean_k030",
        "hawkes_bsi_norm_std_k030",
        "hawkes_bsi_norm_range_k030",
        "hawkes_bsi_norm_positive_share_k030",
    ]
    if column in hawkes_bsi_features.columns
]

fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
hawkes_bsi_features[hawkes_plot_cols].plot(ax=axes[0])
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Hawkes BSI Normalized Order-Flow Pressure by Venue")

hawkes_bsi_features[aggregate_cols].plot(ax=axes[1])
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Cross-Venue Hawkes BSI Agreement and Dispersion")

fig.autofmt_xdate(); plt.tight_layout()

## Feature Matrix

In [ ]:
feature_matrix.tail(20)

In [ ]:
selected = [
    column for column in feature_matrix.columns
    if column in ["reference_price", "rv_5m", "bpv_5m", "jump_share_5m", "term_slope_30_90_minus_0_7", "smile_risk_reversal_proxy", "basis_fut_30_90d"]
]
feature_matrix[selected].plot(figsize=(15, max(4, 2 * len(selected))), subplots=True, layout=(len(selected), 1), sharex=True, legend=True)
plt.tight_layout()

## Optional Save

In [ ]:
if SAVE_OUTPUTS:
    out_dir = FEATURE_ROOT / DATE / f"bar_{BAR_MINUTES}m"
    out_dir.mkdir(parents=True, exist_ok=True)
    feature_matrix.to_parquet(out_dir / "feature_matrix.parquet")
    base_feature_matrix.to_parquet(out_dir / "base_feature_matrix.parquet")
    print(f"wrote features to {out_dir}")
else:
    print("SAVE_OUTPUTS is false; nothing written")